### Hard Voting

In [5]:
import pandas as pd

In [2]:
fold1 = pd.read_csv('baseline2_fold1.csv')
fold2 = pd.read_csv('baseline2_fold2.csv')
fold3 = pd.read_csv('baseline2_fold3.csv')
fold4= pd.read_csv('baseline2_fold4.csv')
fold5 = pd.read_csv('baseline2_fold5.csv')

In [3]:
# 'label' 열을 가져와서 데이터프레임으로 변환
labels_df = pd.DataFrame({
    'fold1': fold1['label'],
    'fold2': fold2['label'],
    'fold3': fold3['label'],
    'fold4': fold4['label'],
    'fold5': fold5['label']
})

# 행별로 가장 많이 등장하는 라벨 선택 (다수결 투표)
hard_voting_labels = labels_df.mode(axis=1)[0]

In [4]:
# 결과 저장
hard_voting_df = fold1.copy()
hard_voting_df['label'] = hard_voting_labels

# 하드 보팅 결과 저장
hard_voting_df.to_csv('result2.csv', index=False)

In [19]:
print(len(fold1[fold1['label'] == hard_voting_df['label']]))
print(len(fold1[fold2['label'] == hard_voting_df['label']]))
print(len(fold1[fold3['label'] == hard_voting_df['label']]))
print(len(fold1[fold4['label'] == hard_voting_df['label']]))
print(len(fold1[fold5['label'] == hard_voting_df['label']]))

238
241
237
239
226


### Soft Voting

In [55]:
import pandas as pd
import numpy as np
from ast import literal_eval
# literal_eval : str -> 리스트 
fold_paths = [
    'baseline5_fold0.csv',
    'baseline5_fold1.csv',
    'baseline5_fold2.csv',
    'baseline5_fold3.csv',
    'baseline5_fold4.csv'
]
folds = [pd.read_csv(path, converters={'probs': literal_eval}) for path in fold_paths]
probs_stack = np.array([df['probs'].tolist() for df in folds])  # shape: (5, num_samples, num_classes)

In [69]:
# Soft Voting: 클래스별 평균 확률 계산
soft_voted_probs = np.mean(probs_stack, axis=0)  
# 최종 예측 클래스 결정
soft_voted_preds = np.argmax(soft_voted_probs, axis=1)

In [ ]:
# encoder classes
labels = {0:'airplane', 1:'apple' , 2:'ball' , 3:'bird', 4:'building', 5:'cat', 6: 'emotion_face',
 7:'police_car', 8:'rabbit', 9:'truck'}

soft_voting_df = folds[0][['ID']].copy()  # ID 컬럼 유지
soft_voting_df['label'] = soft_voted_preds
soft_voting_df['label'] = soft_voting_df['label'].map(labels)

# save
soft_voting_df.to_csv("results-1.csv", index=False)